In [3]:
# ============================================================
# CELDA 0+1 — CONFIGURACIÓN + CAPTURA (VERSIÓN REPARADA)
# ============================================================

import os
import cv2
import numpy as np
from pathlib import Path

# ========= RUTAS QUE ME DISTE =========
BASE_DIR = Path(r"C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc")
DATA_DIR = BASE_DIR / "fotos"
MODEL_PATH = BASE_DIR / "modelo_cnn_rostros.keras"
LABELS_PATH = BASE_DIR / "labels_indices.json"

IMG_SIZE = 160

BASE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("📁 BASE_DIR:", BASE_DIR)
print("📁 DATA_DIR:", DATA_DIR)

# ========= DETECTOR DE CARAS =========
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    raise RuntimeError("No se cargó HaarCascade.")

# ========= NOMBRE DE CLASE =========
person_name = input("Nombre de la persona: ").strip()
if not person_name:
    raise ValueError("Nombre inválido.")

person_dir = DATA_DIR / person_name
person_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📸 Se guardarán imágenes en: {person_dir}")
print("Instrucciones:")
print("  - Presiona 'C' para tomar foto")
print("  - Presiona 'Q' o 'ESC' para salir\n")

# ========= ABRIR CÁMARA =========
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara.")

img_count = 0

def tecla_presionada(k):
    return k in [ord('c'), ord('C'), ord('q'), ord('Q'), 27]

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("⚠️ No se pudo leer frame.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(80, 80)
        )

        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        cv2.putText(frame, f"{person_name} | Fotos: {img_count}",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

        cv2.imshow("Captura Rostros (C=guardar, Q/ESC=salir)", frame)

        key = cv2.waitKey(1) & 0xFF

        # ======= GUARDAR FOTO =======
        if key == ord('c') or key == ord('C'):
            if len(faces) == 0:
                print("❌ No se detectó rostro.")
                continue

            (x, y, w, h) = faces[0]

            # Validar ROI
            if w <= 0 or h <= 0:
                print("❌ ROI inválido.")
                continue
            
            face_roi = frame[y:y+h, x:x+w]

            if face_roi.size == 0:
                print("❌ ROI vacío.")
                continue

            face_resized = cv2.resize(face_roi, (IMG_SIZE, IMG_SIZE))

            filename = person_dir / f"{person_name}_{img_count:04d}.jpg"
            saved = cv2.imwrite(str(filename), face_resized)

            if saved:
                img_count += 1
                print(f"✔️ Foto guardada: {filename}")
            else:
                print("❌ Error al guardar la imagen.")

        # ======= SALIR =======
        if key == ord('q') or key == ord('Q') or key == 27:
            print("🔚 Saliendo...")
            break

finally:
    cap.release()
    cv2.destroyAllWindows()

print(f"\n📸 Total de fotos tomadas para '{person_name}': {img_count}")


📁 BASE_DIR: C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc
📁 DATA_DIR: C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc\fotos



📸 Se guardarán imágenes en: C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc\fotos\Axel Alarcon
Instrucciones:
  - Presiona 'C' para tomar foto
  - Presiona 'Q' o 'ESC' para salir

🔚 Saliendo...

📸 Total de fotos tomadas para 'Axel Alarcon': 0


In [ ]:
# ============================================
# CELDA 2 - ENTRENAMIENTO DEL MODELO CNN
# ============================================

# Comprobamos que haya subcarpetas (clases) en DATA_DIR
classes = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
print("Clases encontradas:", classes)
if len(classes) < 2:
    raise RuntimeError("Se necesitan al menos 2 clases para entrenar. Captura más rostros en CELDA 1.")

BATCH_SIZE = 16
EPOCHS = 15  # Puedes subirlo si tienes tiempo/dataset grande

# Generadores de datos con augmentación
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    directory=str(DATA_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory(
    directory=str(DATA_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

num_classes = train_generator.num_classes
print("Número de clases:", num_classes)
print("class_indices:", train_generator.class_indices)

# Guardamos el índice de clases en un JSON
with open(LABELS_PATH, "w", encoding="utf-8") as f:
    json.dump(train_generator.class_indices, f, ensure_ascii=False, indent=4)
print(f"Índices de clases guardados en: {LABELS_PATH}")

# ====== Construcción del modelo (Transfer Learning con MobileNetV2) ======
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar capas base (al inicio)
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
preds = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ====== Entrenamiento ======
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator
)

# ====== (Opcional) Descongelar últimas capas para fine-tuning ======
# Descomenta este bloque si quieres afinar más el modelo luego de primeras épocas.
"""
for layer in base_model.layers[-40:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator
)
"""

# ====== Graficar accuracy y loss ======
plt.figure()
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Accuracy')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.show()

# ====== Guardar modelo entrenado ======
model.save(MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")


In [ ]:
# ============================================
# CELDA 3 - RECONOCIMIENTO EN TIEMPO REAL
# ============================================

# Cargar modelo entrenado
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo en {MODEL_PATH}. Entrena el modelo en la CELDA 2 primero.")

model = load_model(MODEL_PATH)
print("Modelo cargado desde:", MODEL_PATH)

# Cargar mapeo de clases
if not LABELS_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo de labels en {LABELS_PATH}. Entrena el modelo en la CELDA 2 primero.")

with open(LABELS_PATH, "r", encoding="utf-8") as f:
    class_indices = json.load(f)

# Invertir el diccionario: índice -> nombre_clase
idx_to_class = {v: k for k, v in class_indices.items()}
print("Clases:", idx_to_class)

# Detector de rostros
cascade_path = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    raise RuntimeError("No se pudo cargar el clasificador de rostros para reconocimiento.")

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara para reconocimiento.")

print("Reconocimiento en tiempo real iniciado. Presiona 'q' para salir.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("No se pudo leer frame de la cámara.")
            break

        # Convertimos a escala de grises solo para detección
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5,
            minSize=(60, 60)
        )

        for (x, y, w, h) in faces:
            face_roi = frame[y:y + h, x:x + w]

            # Preprocesar para el modelo
            face_resized = cv2.resize(face_roi, (IMG_SIZE, IMG_SIZE))
            face_array = face_resized.astype("float32") / 255.0
            face_array = np.expand_dims(face_array, axis=0)  # (1, H, W, 3)

            # Predicción
            preds = model.predict(face_array, verbose=0)[0]
            max_idx = np.argmax(preds)
            prob = preds[max_idx]
            label = idx_to_class.get(max_idx, "desconocido")

            text = f"{label}: {prob*100:.1f}%"

            # Dibujar bounding box y etiqueta
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.rectangle(frame, (x, y - 25), (x + w, y), (0, 255, 0), cv2.FILLED)
            cv2.putText(frame, text, (x + 5, y - 7),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        cv2.imshow("Reconocimiento facial CNN - Presiona 'q' para salir", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()

print("Reconocimiento finalizado.")
